# Natural Language Processing Coursework

## Setup

In [1]:


import os
if not os.path.exists("NLPLabs-2024"):
    !git clone -q https://github.com/CRLala/NLPLabs-2024.git


if not os.path.exists("dontpatronizeme"):
    !git clone -q https://github.com/Perez-AlmendrosC/dontpatronizeme.git

    
pcl_tsv_path = "NLPLabs-2024/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
train_split_path = "dontpatronizeme/semeval-2022/practice splits/train_semeval_parids-labels.csv"
dev_split_path   = "dontpatronizeme/semeval-2022/practice splits/dev_semeval_parids-labels.csv"


In [2]:
import os
from pathlib import Path
import re
import csv
import numpy as np
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

In [3]:
from huggingface_hub import login
import os
# Option 1: token from environment variable (recommended)
token = os.getenv("HF_TOKEN")
if token is not None:
    login(token=token)
else:
    # Interactive login (will prompt in notebook / terminal)
    login()

In [4]:
rows=[]
with open(pcl_tsv_path) as f:
    for line in f.readlines()[4:]:
        par_id=line.strip().split('\t')[0]
        art_id = line.strip().split('\t')[1]
        keyword=line.strip().split('\t')[2]
        country=line.strip().split('\t')[3]
        t=line.strip().split('\t')[4]#.lower()
        l=line.strip().split('\t')[-1]
        if l=='0' or l=='1':
            lbin=0
        else:
            lbin=1
        rows.append(
            {'par_id':int(par_id),
            'doc_id':art_id,
            'keyword':keyword,
            'country':country,
            'text':t, 
            'label':lbin, 
            'orig_label':int(l)
            }
        )
df=pd.DataFrame(rows, columns=['par_id', 'doc_id', 'keyword', 'country', 'text', 'label', 'orig_label'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10469 entries, 0 to 10468
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   par_id      10469 non-null  int64
 1   doc_id      10469 non-null  str  
 2   keyword     10469 non-null  str  
 3   country     10469 non-null  str  
 4   text        10469 non-null  str  
 5   label       10469 non-null  int64
 6   orig_label  10469 non-null  int64
dtypes: int64(3), str(4)
memory usage: 3.4 MB


In [5]:
import ast
import numpy as np
import pandas as pd


train_split = pd.read_csv(train_split_path)
dev_split   = pd.read_csv(dev_split_path)

# Parse 7-dim multi-hot label vectors
def parse_label_vec(x):
    v = ast.literal_eval(x) if isinstance(x, str) else x
    if not isinstance(v, list) or len(v) != 7:
        raise ValueError(f"Expected 7-dim list, got: {x}")
    # float for BCEWithLogitsLoss in multi-label training
    return [float(t) for t in v]

train_split["par_id"] = train_split["par_id"].astype(int)
dev_split["par_id"]   = dev_split["par_id"].astype(int)
train_split["label_vec"] = train_split["label"].apply(parse_label_vec)
dev_split["label_vec"]   = dev_split["label"].apply(parse_label_vec)

# Build train/dev dataframes based on par_id
train_ids = set(train_split["par_id"].tolist())
dev_ids   = set(dev_split["par_id"].tolist())

df_train = df[df["par_id"].isin(train_ids)].copy().reset_index(drop=True)
df_dev   = df[df["par_id"].isin(dev_ids)].copy().reset_index(drop=True)

# Attach label_vec
train_map = dict(zip(train_split["par_id"], train_split["label_vec"]))
dev_map   = dict(zip(dev_split["par_id"], dev_split["label_vec"]))

df_train["label_vec"] = df_train["par_id"].map(train_map)
df_dev["label_vec"]   = df_dev["par_id"].map(dev_map)

# Sanity checks
assert df_train["label_vec"].notna().all(), "Some train rows missing label_vec (par_id mismatch)"
assert df_dev["label_vec"].notna().all(), "Some dev rows missing label_vec (par_id mismatch)"
assert set(df_train["label"].unique()).issubset({0, 1}), "Binary label column must be 0/1"
assert set(df_dev["label"].unique()).issubset({0, 1}), "Binary label column must be 0/1"

print("Train:", df_train.shape, "pos_rate:", df_train["label"].mean())
print("Dev:  ", df_dev.shape,   "pos_rate:", df_dev["label"].mean())

Train: (8375, 8) pos_rate: 0.09480597014925374
Dev:   (2094, 8) pos_rate: 0.09503342884431709


In [6]:
import numpy as np
import pandas as pd
import random
import torch

from sklearn.metrics import average_precision_score, f1_score

TEST_TSV = "dontpatronizeme/semeval-2022/TEST/task4_test.tsv"

# ---- Training config
MODEL_NAME = "roberta-base"
MAX_LEN = 256

VAL_FRAC = 0.10                 # internal val split size
SEEDS = [42, 43, 44]            # ensemble seeds (high-yield improvement #1)
TASK1_GRID = [(2e-5, 4), (3e-5, 4)]  # (lr, epochs) tiny tune (high-yield improvement #2)
USE_KEYWORD_BALANCE = False      # high-yield improvement #3

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def stratified_split(df: pd.DataFrame, label_col="label", val_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    pos_idx = df.index[df[label_col] == 1].to_numpy().copy()
    neg_idx = df.index[df[label_col] == 0].to_numpy().copy()
    rng.shuffle(pos_idx)
    rng.shuffle(neg_idx)

    npos = max(1, int(round(len(pos_idx) * val_frac))) if len(pos_idx) else 0
    nneg = max(1, int(round(len(neg_idx) * val_frac))) if len(neg_idx) else 0

    val_idx = np.concatenate([pos_idx[:npos], neg_idx[:nneg]])
    train_idx = np.setdiff1d(df.index.to_numpy(), val_idx, assume_unique=False)

    return df.loc[train_idx].reset_index(drop=True), df.loc[val_idx].reset_index(drop=True)

In [7]:
def load_pcl_tsv(path: str, has_label: bool):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or line.startswith("-"):
                continue
            parts = line.split("\t")
            if len(parts) < 5:
                continue
            if not parts[0].isdigit():
                continue

            par_id = int(parts[0])
            art_id = parts[1]
            keyword = parts[2]
            country = parts[3]

            if has_label:
                text = "\t".join(parts[4:-1]).strip()
                label = int(parts[-1])
            else:
                text = "\t".join(parts[4:]).strip()
                label = None

            rows.append({"par_id": par_id, "art_id": art_id, "keyword": keyword, "country": country, "text": text, "label": label})
    out = pd.DataFrame(rows)
    assert len(out) > 0, f"Parsed 0 rows from {path}"
    return out

def reorder_by_parid(df: pd.DataFrame, parids_in_order):
    order = {int(pid): i for i, pid in enumerate(parids_in_order)}
    out = df[df["par_id"].isin(order.keys())].copy()
    out["__o"] = out["par_id"].map(order)
    out = out.sort_values("__o").drop(columns="__o").reset_index(drop=True)
    return out

# 4. Model

In [8]:
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import Trainer

def keyword_invfreq_weights(df: pd.DataFrame, col="keyword"):
    if col not in df.columns:
        return None
    freq = df[col].value_counts().to_dict()
    w = df[col].map(lambda k: 1.0 / float(freq.get(k, 1))).to_numpy(dtype=np.float64)
    w = w / np.mean(w)  # normalise scale
    return w

class StripKeysCollator:
    def __init__(self, base_collator, drop=("keyword",)):
        self.base = base_collator
        self.drop = set(drop)
    def __call__(self, features):
        for f in features:
            for k in list(f.keys()):
                if k in self.drop:
                    f.pop(k, None)
        return self.base(features)
class WeightedCEKeywordTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, sampler_weights: np.ndarray | None = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_cpu = class_weights.detach().cpu()
        self.loss_fct = None  # build lazily on correct device
        self.sampler_weights = sampler_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        # Accept num_items_in_batch / **kwargs for HF Trainer compatibility
        labels = inputs.pop("labels")
        if isinstance(labels, torch.Tensor):
            labels = labels.long()

        outputs = model(**inputs)
        logits = outputs.logits

        # Build loss once on the right device
        if self.loss_fct is None or self.loss_fct.weight.device != logits.device:
            w = self.class_weights_cpu.to(logits.device)
            self.loss_fct = nn.CrossEntropyLoss(weight=w)

        loss = self.loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.sampler_weights is None:
            return super().get_train_dataloader()

        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(self.sampler_weights, dtype=torch.double),
            num_samples=len(self.sampler_weights),
            replacement=True,
        )

        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )
# metrics without threshold sweep
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

def compute_metrics_task1(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    y = labels.astype(int)
    pred05 = (probs >= 0.5).astype(int)
    return {
        "auprc": float(average_precision_score(y, probs)),
        "f1@0.5": float(f1_score(y, pred05, zero_division=0)),
        "precision@0.5": float(precision_score(y, pred05, zero_division=0)),
        "recall@0.5": float(recall_score(y, pred05, zero_division=0)),
    }

def best_f1_threshold(y_true: np.ndarray, probs: np.ndarray, steps: int = 2001):
    grid = np.linspace(0.0, 1.0, steps)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_true, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_t = float(t)
    return best_t, best_f1

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments,
    EarlyStoppingCallback
)

from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and (not use_bf16)
model_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_fp16 else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
base_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if device == "cuda" else None
)

def make_task2_ds(df_, max_len=256):
    ds = Dataset.from_pandas(df_[["text", "label_vec"]], preserve_index=False)
    def tok(b):
        enc = tokenizer(b["text"], truncation=True, max_length=max_len)
        enc["labels"] = b["label_vec"]
        return enc
    return ds.map(tok, batched=True, remove_columns=["text", "label_vec"])

def make_task1_ds(df_, keep_keyword: bool, max_len=256):
    cols = ["text", "label"] + (["keyword"] if keep_keyword else [])
    ds = Dataset.from_pandas(df_[cols], preserve_index=False)
    def tok(b):
        enc = tokenizer(b["text"], truncation=True, max_length=max_len)
        enc["labels"] = b["label"]
        if keep_keyword and "keyword" in b:
            enc["keyword"] = b["keyword"]
        return enc
    remove_cols = ["text", "label"] + (["keyword"] if keep_keyword else [])
    return ds.map(tok, batched=True, remove_columns=remove_cols)

def make_infer_ds(df_, max_len=256):
    ds = Dataset.from_pandas(df_[["text"]], preserve_index=False)
    def tok(b):
        return tokenizer(b["text"], truncation=True, max_length=max_len)
    return ds.map(tok, batched=True, remove_columns=["text"])

@torch.no_grad()
def predict_probs_pos(model, infer_ds, batch_size=16):
    model.eval()
    dl = DataLoader(infer_ds, batch_size=batch_size, shuffle=False, collate_fn=base_collator)
    probs = []
    for batch in dl:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        logits = model(**batch).logits
        p = torch.softmax(logits, dim=-1)[:, 1]
        probs.append(p.detach().cpu())
    return torch.cat(probs).numpy()

def train_one_seed(seed: int, df_train_inner, df_val_inner, class_weights: torch.Tensor):
    set_seed(seed)

    # ---- Task2
    task2_out = f"checkpoints/task2_seed{seed}"
    model_t2 = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=7, problem_type="multi_label_classification"
    ).to(device)

    train_t2 = make_task2_ds(df_train_inner, MAX_LEN)
    val_t2   = make_task2_ds(df_val_inner, MAX_LEN)

    def compute_metrics_task2(eval_pred):
        logits, labels = eval_pred
        probs = 1.0 / (1.0 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        y = labels.astype(int)
        tp = (preds & y).sum()
        fp = (preds & (1 - y)).sum()
        fn = ((1 - preds) & y).sum()
        prec = tp / (tp + fp + 1e-12)
        rec  = tp / (tp + fn + 1e-12)
        f1   = 2 * prec * rec / (prec + rec + 1e-12)
        auprc_micro = average_precision_score(labels.ravel().astype(int), probs.ravel())

        return {"f1_micro": float(f1), "auprc_micro": float(auprc_micro)}

    args_t2 = TrainingArguments(
        output_dir=task2_out,
        seed=seed,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.06,
        eval_strategy="no",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="auprc_micro",
        greater_is_better=False,
        save_total_limit=1,
        report_to="none",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=False,
        eval_accumulation_steps=16,
    )

    t2 = Trainer(
        model=model_t2,
        args=args_t2,
        train_dataset=train_t2,
        eval_dataset=val_t2,
        data_collator=base_collator,
        compute_metrics=compute_metrics_task2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )
    t2.train()

    try:
        t2.model.save_pretrained(task2_out, safe_serialization=True)
    except TypeError:
        t2.model.save_pretrained(task2_out)
    tokenizer.save_pretrained(task2_out)

    # ---- Task1: tiny grid search on inner-val AUPRC
    best = None

    for lr, epochs in TASK1_GRID:
        run_dir = f"checkpoints/task1_seed{seed}_lr{lr}_ep{epochs}"

        train_t1 = make_task1_ds(df_train_inner, keep_keyword=True,  max_len=MAX_LEN)
        val_t1   = make_task1_ds(df_val_inner,   keep_keyword=False, max_len=MAX_LEN)

        sampler_w = keyword_invfreq_weights(df_train_inner) if USE_KEYWORD_BALANCE else None

        model_t1 = AutoModelForSequenceClassification.from_pretrained(
            task2_out, num_labels=2, ignore_mismatched_sizes=True
        ).to(device)

        args_t1 = TrainingArguments(
            output_dir=run_dir,
            seed=seed,
            num_train_epochs=epochs,
            learning_rate=lr,
            weight_decay=0.01,
            warmup_ratio=0.06,
            eval_strategy="no",
            save_strategy="no",
            load_best_model_at_end=False,
            metric_for_best_model="auprc",
            greater_is_better=True,
            save_total_limit=1,
            remove_unused_columns=False,  # keep keyword column for sampler, collator strips it
            report_to="none",
            per_device_train_batch_size=8,
            per_device_eval_batch_size=16,
            gradient_accumulation_steps=2,
            fp16=use_fp16,
            bf16=use_bf16,
            gradient_checkpointing=False,
            eval_accumulation_steps=16,
        )

        t1 = WeightedCEKeywordTrainer(
            class_weights=class_weights,
            sampler_weights=sampler_w,
            model=model_t1,
            args=args_t1,
            train_dataset=train_t1,
            eval_dataset=val_t1,
            data_collator=StripKeysCollator(base_collator, drop=("keyword",)),
            compute_metrics=compute_metrics_task1,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
        )
        t1.train()

        try:
            t1.model.save_pretrained(run_dir, safe_serialization=True)
        except TypeError:
            t1.model.save_pretrained(run_dir)
        tokenizer.save_pretrained(run_dir)

        pred = t1.predict(val_t1)
        probs_val = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
        y_val = pred.label_ids.astype(int)
        auprc = float(average_precision_score(y_val, probs_val))

        if (best is None) or (auprc > best["auprc"]):
            best = {
                "dir": run_dir,
                "probs": probs_val,
                "y": y_val,
                "auprc": auprc,
                "lr": lr,
                "epochs": epochs,
            }

    return best["dir"], best["probs"], best["y"], best["auprc"], best["lr"], best["epochs"]

In [10]:
def refit_one_seed(seed: int, lr: float, epochs: int, class_weights: torch.Tensor):
    set_seed(seed)

    # ---- Task2 refit on FULL df_train (no eval, no early stopping)
    task2_out = f"checkpoints/task2_refit_seed{seed}"
    model_t2 = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=7, problem_type="multi_label_classification"
    ).to(device)

    train_t2 = make_task2_ds(df_train, MAX_LEN)

    args_t2 = TrainingArguments(
        output_dir=task2_out,
        seed=seed,
        num_train_epochs=3,
        learning_rate=2e-5,
        # per_device_train_batch_size=16,
        # per_device_eval_batch_size=32,
        weight_decay=0.01,
        warmup_ratio=0.06,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        eval_accumulation_steps=16,
    )

    t2 = Trainer(
        model=model_t2,
        args=args_t2,
        train_dataset=train_t2,
        data_collator=base_collator,
    )
    t2.train()
    try:
        t2.model.save_pretrained(task2_out, safe_serialization=True)
    except TypeError:
        t2.model.save_pretrained(task2_out)
    tokenizer.save_pretrained(task2_out)

    # ---- Task1 refit on FULL df_train using chosen (lr, epochs)
    task1_out = f"checkpoints/task1_refit_seed{seed}_lr{lr}_ep{epochs}"

    train_t1 = make_task1_ds(df_train, keep_keyword=True, max_len=MAX_LEN)

    sampler_w = keyword_invfreq_weights(df_train) if USE_KEYWORD_BALANCE else None

    model_t1 = AutoModelForSequenceClassification.from_pretrained(
        task2_out, num_labels=2, ignore_mismatched_sizes=True
    ).to(device)

    args_t1 = TrainingArguments(
        output_dir=task1_out,
        seed=seed,
        num_train_epochs=epochs,
        learning_rate=lr,
        # per_device_train_batch_size=16,
        # per_device_eval_batch_size=64,
        weight_decay=0.01,
        warmup_ratio=0.06,
        eval_strategy="no",
        save_strategy="no",
        remove_unused_columns=False,
        report_to="none",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        eval_accumulation_steps=16,
    )

    t1 = WeightedCEKeywordTrainer(
        class_weights=class_weights,
        sampler_weights=sampler_w,
        model=model_t1,
        args=args_t1,
        train_dataset=train_t1,
        data_collator=StripKeysCollator(base_collator, drop=("keyword",)),
        compute_metrics=None,
    )
    t1.train()
    try:
        t1.model.save_pretrained(task1_out, safe_serialization=True)
    except TypeError:
        t1.model.save_pretrained(task1_out)
    tokenizer.save_pretrained(task1_out)

    return task1_out

In [11]:
# ---- Internal split for tuning (dev leakage fix)
df_train_inner, df_val_inner = stratified_split(df_train, val_frac=VAL_FRAC, seed=SEEDS[0])
print("Inner train:", df_train_inner.shape, "pos_rate:", df_train_inner["label"].mean())
print("Inner val:  ", df_val_inner.shape,   "pos_rate:", df_val_inner["label"].mean())

# ---- Class weights from FULL training set
neg = int((df_train["label"] == 0).sum())
pos = int((df_train["label"] == 1).sum())
w_pos = neg / max(pos, 1)
class_weights = torch.tensor([1.0, float(w_pos)], dtype=torch.float32)
print(f"neg={neg} pos={pos} w_pos={w_pos:.4f}")

# ---- Train per seed, pick best run per seed by inner-val AUPRC
best_dirs = []
val_probs = []
val_y = None
val_auprcs = []
best_cfgs = []

for s in SEEDS:
    d, p, y, a, best_lr, best_ep = train_one_seed(s, df_train_inner, df_val_inner, class_weights)
    best_dirs.append(d)
    val_probs.append(p)
    val_auprcs.append(a)
    best_cfgs.append((best_lr, best_ep))
    if val_y is None:
        val_y = y
    else:
        assert np.array_equal(val_y, y)

print("Per-seed best AUPRC:", list(zip(SEEDS, val_auprcs)))

# ---- Ensemble threshold selection on inner-val (leakage-free)
ens_val_probs = np.mean(np.stack(val_probs, axis=0), axis=0)
best_t, best_f1 = best_f1_threshold(val_y, ens_val_probs, steps=2001)
print(f"Chosen threshold from inner-val: t={best_t:.4f}, F1={best_f1:.6f}")

refit_dirs = []
for s, (lr, ep) in zip(SEEDS, best_cfgs):
    refit_dir = refit_one_seed(s, lr, ep, class_weights)
    refit_dirs.append(refit_dir)

print("Refit Task1 dirs:", refit_dirs)

Inner train: (7538, 8) pos_rate: 0.09485274608649509
Inner val:   (837, 8) pos_rate: 0.09438470728793309
neg=7581 pos=794 w_pos=9.5479


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/7538 [00:00<?, ? examples/s]

Map:   0%|          | 0/837 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: checkpoints/task2_refit_seed44
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
500,1.439633
1000,0.879708
1500,0.475822
2000,0.374929


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Refit Task1 dirs: ['checkpoints/task1_refit_seed42_lr2e-05_ep4', 'checkpoints/task1_refit_seed43_lr2e-05_ep4', 'checkpoints/task1_refit_seed44_lr2e-05_ep4']


In [12]:
rows=[]
with open(TEST_TSV) as f:
    for line in f.readlines()[4:]:
        par_id=line.strip().split('\t')[0]
        art_id = line.strip().split('\t')[1]
        keyword=line.strip().split('\t')[2]
        country=line.strip().split('\t')[3]
        t=line.strip().split('\t')[4]#.lower()
        if l=='0' or l=='1':
            lbin=0
        else:
            lbin=1
        rows.append(
            {'par_id':par_id,
            'doc_id':art_id,
            'keyword':keyword,
            'country':country,
            'text':t, 
            }
        )
df_test =pd.DataFrame(rows, columns=['par_id', 'doc_id', 'keyword', 'country', 'text'])
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 3828 entries, 0 to 3827
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   par_id   3828 non-null   str  
 1   doc_id   3828 non-null   str  
 2   keyword  3828 non-null   str  
 3   country  3828 non-null   str  
 4   text     3828 non-null   str  
dtypes: str(5)
memory usage: 1.2 MB


In [13]:
# ---- Load official test data (no auto discovery)
# df_test = load_pcl_tsv(TEST_TSV, has_label=False).reset_index(drop=True)

assert df_test["par_id"].is_unique, "Test TSV has duplicate par_id rows"
assert df_test["text"].notna().all(), "Some test texts are missing"

# ---- Inference datasets
dev_infer = make_infer_ds(df_dev, MAX_LEN)
test_infer = make_infer_ds(df_test, MAX_LEN)

# ---- Ensemble predictions on official dev + official test
dev_probs_seeds = []
test_probs_seeds = []

for d in refit_dirs:
    m = AutoModelForSequenceClassification.from_pretrained(d).to(device)
    dev_probs_seeds.append(predict_probs_pos(m, dev_infer))
    test_probs_seeds.append(predict_probs_pos(m, test_infer))

dev_probs_ens = np.mean(np.stack(dev_probs_seeds, axis=0), axis=0)
test_probs_ens = np.mean(np.stack(test_probs_seeds, axis=0), axis=0)

dev_pred = (dev_probs_ens >= best_t).astype(int)
test_pred = (test_probs_ens >= best_t).astype(int)

# Optional: evaluate dev (not used for tuning)
dev_f1 = f1_score(df_dev["label"].astype(int).to_numpy(), dev_pred, zero_division=0)
print(f"Official dev F1 (not used for selection): {dev_f1:.6f}")

# ---- Write outputs
with open("dev.txt", "w", encoding="utf-8") as f:
    for p in dev_pred.tolist():
        f.write(f"{int(p)}\n")

with open("test.txt", "w", encoding="utf-8") as f:
    for p in test_pred.tolist():
        f.write(f"{int(p)}\n")

print("Wrote dev.txt and test.txt")

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Map:   0%|          | 0/3828 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Official dev F1 (not used for selection): 0.599455
Wrote dev.txt and test.txt
